In [ ]:
import torch
print("GPU Active:", torch.cuda.is_available())

In [ ]:
!pip install transformers datasets accelerate evaluate -q

In [ ]:
import pandas as pd
import re
import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import classification_report


df = pd.read_csv('/content/datasetCrime.csv', on_bad_lines='skip', engine='python')


df = df.dropna(subset=['crimeaditionalinfo', 'category'])
df = df.drop_duplicates()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['crimeaditionalinfo'].apply(clean_text)

def make_binary_category(cat):
    cat = str(cat).lower()
    if 'financial' in cat or 'fraud' in cat:
        return 'Financial Fraud'
    else:
        return 'Non Financial Cybercrime'

df['binary_category'] = df['category'].apply(make_binary_category)


df['label'] = df['binary_category'].map({
    'Financial Fraud': 1,
    'Non Financial Cybercrime': 0
})

In [ ]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    # tokenising the text n fixing the highest length 512 char
    return tokenizer(examples["clean_text"], truncation=True, max_length=512)

# Split the DataFrame into training, validation, and test sets
train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42, stratify=train_val_df['label']) # 0.25 * 0.8 = 0.2

# Convert pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# tokezise the whole
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

#   DistilBERT model load for class
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# eval metrics
clf_metrics = evaluate.combine(["accuracy", "f1"])

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return clf_metrics.compute(predictions=preds, references=labels)

# custom trainer
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # weight for handling imbalance
        loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.5, 1.0]).to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)


trainer.train()

In [ ]:
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=1)

print("--- DistilBERT Transformer Performance ---")
print(classification_report(tokenized_test["label"], preds, target_names=['Non Financial Cybercrime', 'Financial Fraud']))

In [ ]:
import matplotlib.pyplot as plt

# safety net data (from dashboard)
train_loss = [0.336583, 0.291103]
val_loss = [0.353051, 0.353043]
val_accuracy = [0.820611, 0.835983]
val_f1 = [0.842315, 0.863418]

# fxing x axis for 2 epoch (X-axis)
plot_epochs = [1, 2]

# --- graph 1: Training vs Validation Loss ---
plt.figure(figsize=(10, 5))
plt.plot(plot_epochs, train_loss, 'o-', label='Training Loss', color='royalblue', linewidth=2)
plt.plot(plot_epochs, val_loss, 'o--', label='Validation Loss', color='orangered', linewidth=2)
plt.title('DistilBERT: Training vs Validation Loss Across Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.xticks(plot_epochs)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

print("\n") #

# --- graph 2: Validation Accuracy & F1-Score ---
plt.figure(figsize=(10, 5))
plt.plot(plot_epochs, val_accuracy, 's-', label='Validation Accuracy', color='forestgreen', linewidth=2)
plt.plot(plot_epochs, val_f1, 'd-', label='Validation F1-Score', color='darkorchid', linewidth=2)
plt.title('DistilBERT: Validation Accuracy & F1-Score Trend', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Score (0 to 1)', fontsize=12)
plt.xticks(plot_epochs)
plt.ylim(0.75, 0.90)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

In [ ]:
import gc
import torch

#memory cleaning
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoTokenizer

# RoBERTa tokeniser load
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def preprocess_roberta(examples):
    return roberta_tokenizer(examples["clean_text"], truncation=True, max_length=512)

# RoBERTa-new load
tokenized_train_rob = train_dataset.map(preprocess_roberta, batched=True)
tokenized_val_rob = val_dataset.map(preprocess_roberta, batched=True)
tokenized_test_rob = test_dataset.map(preprocess_roberta, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer
import torch

# RoBERTa-base model load
roberta_model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

class RobertaCustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        #
        loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.5, 1.0]).to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import TrainingArguments

roberta_args = TrainingArguments(
    output_dir="./roberta_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,               # 3 epoch
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

roberta_trainer = RobertaCustomTrainer(
    model=roberta_model,
    args=roberta_args,
    train_dataset=tokenized_train_rob,
    eval_dataset=val_dataset_rob if 'val_dataset_rob' in locals() else tokenized_val_rob,
    processing_class=roberta_tokenizer,
    compute_metrics=compute_metrics, #
)

#
roberta_trainer.train()

In [ ]:
# RoBERTa prediction
predictions_rob = roberta_trainer.predict(tokenized_test_rob)
preds_rob = np.argmax(predictions_rob.predictions, axis=1)

from sklearn.metrics import classification_report

print("--- RoBERTa-base Transformer Performance ---")
print(classification_report(tokenized_test_rob["label"], preds_rob, target_names=['Non Financial Cybercrime', 'Financial Fraud']))

In [ ]:
import matplotlib.pyplot as plt

# RoBERTa trainning dashboard data
rob_train_loss = [0.348678, 0.306635, 0.277445]
rob_val_loss = [0.379709, 0.358176, 0.379098]
rob_val_accuracy = [0.812282, 0.840349, 0.838311]
rob_val_f1 = [0.841340, 0.869453, 0.866355]

rob_plot_epochs = [1, 2, 3]

# --- graph 1: RoBERTa Loss Trend ---
plt.figure(figsize=(10, 5))
plt.plot(rob_plot_epochs, rob_train_loss, 'o-', label='Training Loss', color='royalblue', linewidth=2)
plt.plot(rob_plot_epochs, rob_val_loss, 'o--', label='Validation Loss', color='orangered', linewidth=2)
plt.title('RoBERTa-base: Training vs Validation Loss Across Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.xticks(rob_plot_epochs)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

print("\n")

# --- graph 2: RoBERTa Accuracy & F1-Score ---
plt.figure(figsize=(10, 5))
plt.plot(rob_plot_epochs, rob_val_accuracy, 's-', label='Validation Accuracy', color='forestgreen', linewidth=2)
plt.plot(rob_plot_epochs, rob_val_f1, 'd-', label='Validation F1-Score', color='darkorchid', linewidth=2)
plt.title('RoBERTa-base: Validation Accuracy & F1-Score Trend', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Score (0 to 1)', fontsize=12)
plt.xticks(rob_plot_epochs)
plt.ylim(0.75, 0.90)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define model names and their performance metrics from tests
models = ['DistilBERT', 'RoBERTa-base']
accuracy_scores = [0.84, 0.84]
f1_scores = [0.83, 0.83]
non_fin_precision = [0.77, 0.79] # Added to highlight RoBERTa's precision boost in Minority Class

x = np.arange(len(models))  # Set the label locations
width = 0.25  # Set the width of the bars

fig, ax = plt.subplots(figsize=(10, 6))

# Plot bars for each metric side by side
rects1 = ax.bar(x - width, accuracy_scores, width, label='Overall Accuracy', color='teal')
rects2 = ax.bar(x, f1_scores, width, label='Macro Avg F1-Score', color='indianred')
rects3 = ax.bar(x + width, non_fin_precision, width, label='Non-Financial Precision', color='goldenrod')

# Customize chart styling, labels, and titles
ax.set_ylabel('Scores (0.0 to 1.0)', fontsize=12, fontweight='bold')
ax.set_title('Transformer Models Performance Comparison', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11, fontweight='bold')
ax.set_ylim(0.70, 0.90) # Adjusted limits to visually highlight subtle differences clearly
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(loc='lower left', fontsize=11)

# Function to automatically attach text labels on top of each bar
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

# Apply labels to all plotted bar structures
autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

plt.tight_layout()
plt.show()